# Historical Airport Weather Ingestion Using Open-Meteo

This notebook pulls historical hourly weather data from the Open-Meteo Archive API for U.S. airports listed in a CSV file.

## Workflow
- Read airport latitude and longitude data from a CSV file
- Process airports in batches of 50
- Call the Open-Meteo historical archive API for each airport
- Save each airport's hourly weather data as a separate parquet file
- Save a batch log showing which airports succeeded, failed, or were skipped

## Why this workflow?
The free Open-Meteo API has usage limits, so the ingestion is designed to:
- run in smaller batches,
- save progress after each airport,
- avoid repeating completed work,
- make it easy to continue after wifi networks

In [142]:
# ============================================================
# Step 1: Install required packages
# ------------------------------------------------------------
# Run this cell only if the packages are not already installed
# on your local machine.
# ============================================================

#pip install openmeteo-requests
#!pip install requests-cache retry-requests numpy pandas pyarrow fastparquet

In [143]:
# ============================================================
# Step 2: Import libraries
# ------------------------------------------------------------
# This cell imports all packages needed for:
# - file handling
# - API requests
# - retry logic
# - data processing
# - parquet export
# ============================================================

import os
import time
from pathlib import Path

import pandas as pd
import requests_cache
import openmeteo_requests
from retry_requests import retry

In [144]:
# ============================================================
# Step 3: Define project paths
# ------------------------------------------------------------
# This cell defines the root project folder as the folder the file is currently in. 
# We also set the directory for the subfoldes which will hold batch logs and parquet files.
# ============================================================

ROOT_DIR =  Path.cwd()

AIRPORT_DATA_DIR =  ROOT_DIR / "airport_data"
WEATHER_OUTPUT_DIR = ROOT_DIR / "weather_parquet"
BATCH_LOG_DIR = ROOT_DIR / "batch_logs"

# Update this filename if your airport CSV has a different name
AIRPORT_CSV_FILE = AIRPORT_DATA_DIR / "b4178d0f-1e96-4fcb-ad32-cf86292f1238.csv"

print("Root directory:", ROOT_DIR)
print("Airport CSV:", AIRPORT_CSV_FILE)
print("Weather output folder:", WEATHER_OUTPUT_DIR)
print("Batch log folder:", BATCH_LOG_DIR)

Root directory: c:\Users\thanu\OneDrive\Documents\MDSA Fall 2025 to Winter 2026\Data 608\Data 608 Project\Open-Meto-Weather
Airport CSV: c:\Users\thanu\OneDrive\Documents\MDSA Fall 2025 to Winter 2026\Data 608\Data 608 Project\Open-Meto-Weather\airport_data\b4178d0f-1e96-4fcb-ad32-cf86292f1238.csv
Weather output folder: c:\Users\thanu\OneDrive\Documents\MDSA Fall 2025 to Winter 2026\Data 608\Data 608 Project\Open-Meto-Weather\weather_parquet
Batch log folder: c:\Users\thanu\OneDrive\Documents\MDSA Fall 2025 to Winter 2026\Data 608\Data 608 Project\Open-Meto-Weather\batch_logs


## User Settings

This section controls:
- the date range for historical weather data,
- the batch range of airports to process,
- whether existing parquet files should be overwritten,
- the delay between API calls.

In [ ]:
# ============================================================
# Step 4: User settings
# ------------------------------------------------------------
# Change these values when moving from one batch to the next.
#
# Example:
# - First run:  BATCH_START = 0
# - Second run: BATCH_START = 50
# - Third run:  BATCH_START = 100
# ============================================================

START_DATE = "2022-10-29"
END_DATE = "2025-11-03"

BATCH_START = 0
BATCH_SIZE = 50

# Set to False so completed airports are skipped on reruns
OVERWRITE_EXISTING = False

# Small delay between requests to reduce the chance of rate-limit issues
SLEEP_SECONDS = 2

# Weather variables to request from Open-Meteo
HOURLY_VARIABLES = [
    "temperature_2m",
    "wind_speed_10m",
    "wind_gusts_10m",
    "precipitation",
    "rain",
    "snowfall",
    "weather_code"
]

print(f"Batch start: {BATCH_START}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Date range: {START_DATE} to {END_DATE}")

Batch start: 350
Batch size: 50
Date range: 2022-10-29 to 2025-11-03


In [146]:
# ============================================================
# Step 5: Set up the Open-Meteo API client
# ------------------------------------------------------------
# This cell creates a cached session and retry-enabled client.
#
# Why use cache?
# - prevents repeated downloads of the same request during testing
#
# Why use retry?
# - helps handle temporary request failures more gracefully
# ============================================================

cache_session = requests_cache.CachedSession(".cache", expire_after=-1)
retry_session = retry(cache_session, retries=2, backoff_factor=0.3)
openmeteo = openmeteo_requests.Client(session=retry_session)

OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"

print("Open-Meteo client initialized.")

Open-Meteo client initialized.


## Load and inspect airport data

This section reads the airport CSV, checks the columns, and prepares the airport list for batch processing.

In [147]:
# ============================================================
# Step 6: Read the airport CSV
# ------------------------------------------------------------
# This cell loads the airport dataset into pandas so we can
# inspect the column names and structure before cleaning.
# ============================================================

airports = pd.read_csv(AIRPORT_CSV_FILE)

print("Shape of raw airport file:", airports_raw.shape)
print("\nColumn names:")
print(airports.columns.tolist())

airports.head()

Shape of raw airport file: (366, 8)

Column names:
['airport', 'display_airport_name', 'display_city_market_name_full', 'airport_country_name', 'latitude', 'longitude', 'utc_local_time_variation', 'airport_is_latest']


,airport,display_airport_name,display_city_market_name_full,airport_country_name,latitude,longitude,utc_local_time_variation,airport_is_latest
0,ABE,Lehigh Valley International,"Allentown/Bethlehem/Easton, PA",United States,40.651667,-75.442778,-500,1
1,ABI,Abilene Regional,"Abilene, TX",United States,32.408333,-99.682500,-600,1
2,ABQ,Albuquerque International Sunport,"Albuquerque, NM",United States,35.038056,-106.610000,-700,1
3,ABR,Aberdeen Regional,"Aberdeen, SD",United States,45.446667,-98.423056,-600,1
4,ABY,Southwest Georgia Regional,"Albany, GA",United States,31.535556,-84.194444,-500,1


In [148]:
# ============================================================
# Step 7: Keep only the columns needed for ingestion
# ------------------------------------------------------------
# This cell narrows the airport file down to the fields needed
# for API requests and metadata storage.
# ============================================================

required_columns = [
    "airport",
    "airport_name",
    "city_market_name",
    "country_name",
    "latitude",
    "longitude",
    "utc_local_time_variation"
]

# Keep only columns that actually exist in the dataframe
existing_required_columns = [col for col in required_columns if col in airports.columns]
airports = airports[existing_required_columns].copy()

print("Columns kept for processing:")
print(airports.columns.tolist())
airports.head()

Columns kept for processing:
['airport', 'latitude', 'longitude', 'utc_local_time_variation']


,airport,latitude,longitude,utc_local_time_variation
0,ABE,40.651667,-75.442778,-500
1,ABI,32.408333,-99.682500,-600
2,ABQ,35.038056,-106.610000,-700
3,ABR,45.446667,-98.423056,-600
4,ABY,31.535556,-84.194444,-500


In [149]:
# ============================================================
# Step 8: Clean the airport data
# ------------------------------------------------------------
# This cell:
# - removes rows missing airport code or coordinates
# - removes duplicate airport codes
# - resets the index
# - adds a row number to support batch slicing
# ============================================================

airports = airports.dropna(subset=["airport", "latitude", "longitude"]).copy()

# Convert coordinates to numeric in case they were read as strings
airports["latitude"] = pd.to_numeric(airports["latitude"], errors="coerce")
airports["longitude"] = pd.to_numeric(airports["longitude"], errors="coerce")

airports = airports.dropna(subset=["latitude", "longitude"]).copy()

# Keep one row per airport code
airports = airports.drop_duplicates(subset=["airport"]).reset_index(drop=True)

# Add row number for easier batch processing
airports["row_num"] = range(len(airports))

print("Cleaned airport dataset shape:", airports.shape)
airports.head()

Cleaned airport dataset shape: (366, 5)


,airport,latitude,longitude,utc_local_time_variation,row_num
0,ABE,40.651667,-75.442778,-500,0
1,ABI,32.408333,-99.682500,-600,1
2,ABQ,35.038056,-106.610000,-700,2
3,ABR,45.446667,-98.423056,-600,3
4,ABY,31.535556,-84.194444,-500,4


In [150]:
# ============================================================
# Step 9: Select the current batch of airports
# ------------------------------------------------------------
# This cell slices the airport table so only the current batch
# is processed in this notebook run.
# ============================================================

batch_df = airports.iloc[BATCH_START:BATCH_START + BATCH_SIZE].copy()

print(f"Processing {len(batch_df)} airports in this batch.")
print(f"Row range: {BATCH_START} to {BATCH_START + len(batch_df) - 1}")

batch_df[["row_num", "airport", "latitude", "longitude"]].head(10)

Processing 16 airports in this batch.
Row range: 350 to 365


,row_num,airport,latitude,longitude
350,350,TWF,42.480833,-114.485556
351,351,TXK,33.456389,-93.989167
352,352,TYR,32.353889,-95.402778
353,353,TYS,35.811111,-83.994167
354,354,USA,35.387778,-80.709167
355,355,VCT,28.854167,-96.918611
356,356,VEL,40.436111,-109.511389
357,357,VLD,30.790000,-83.275000
358,358,VPS,30.483333,-86.526111
359,359,WRG,56.484444,-132.369722


## Define helper functions

This section defines reusable functions for:
- downloading weather data for one airport,
- converting the response to a pandas DataFrame,
- saving one parquet file per airport.

In [151]:
# ============================================================
# Step 10: Function to fetch weather data for one airport
# ------------------------------------------------------------
# This function:
# - sends the API request for one airport
# - extracts the hourly variables in the correct order
# - converts the result into a pandas DataFrame
# - appends airport metadata to the output
# ============================================================

def fetch_airport_weather(airport_code, latitude, longitude, start_date, end_date):

    params = {
        "latitude": float(latitude),
        "longitude": float(longitude),
        "start_date": start_date,
        "end_date": end_date,
        "hourly": HOURLY_VARIABLES,
        "timezone": "auto"
    }

    responses = openmeteo.weather_api(OPEN_METEO_URL, params=params)
    response = responses[0]

    hourly = response.Hourly()

    # --------------------------------------------------------
    # 1. Build UTC timeline
    # --------------------------------------------------------
    utc_times = pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    )

    # --------------------------------------------------------
    # 2. Convert UTC → local airport time
    # --------------------------------------------------------
    utc_offset = response.UtcOffsetSeconds()

    local_times = (
        utc_times + pd.to_timedelta(utc_offset, unit="s")
    ).tz_localize(None)

    # Remove timezone info for UTC column so parquet is clean
    utc_times = utc_times.tz_localize(None)

    # --------------------------------------------------------
    # 3. Build dataframe
    # --------------------------------------------------------
    hourly_df = pd.DataFrame({
        "datetime_utc": utc_times,
        "datetime_local": local_times,

        "datetime_utc_str": utc_times.strftime("%Y-%m-%d %H:%M:%S"),
        "datetime_local_str": local_times.strftime("%Y-%m-%d %H:%M:%S"),

        "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
        "wind_speed_10m": hourly.Variables(1).ValuesAsNumpy(),
        "wind_gusts_10m": hourly.Variables(2).ValuesAsNumpy(),
        "precipitation": hourly.Variables(3).ValuesAsNumpy(),
        "rain": hourly.Variables(4).ValuesAsNumpy(),
        "snowfall": hourly.Variables(5).ValuesAsNumpy(),
        "weather_code": hourly.Variables(6).ValuesAsNumpy()
    })

    # --------------------------------------------------------
    # 4. Add airport metadata
    # --------------------------------------------------------
    hourly_df["airport"] = airport_code
    hourly_df["latitude"] = float(latitude)
    hourly_df["longitude"] = float(longitude)

    hourly_df["timezone"] = response.Timezone()
    hourly_df["timezone_abbreviation"] = response.TimezoneAbbreviation()
    hourly_df["utc_offset_seconds"] = response.UtcOffsetSeconds()
    hourly_df["elevation"] = response.Elevation()

    return hourly_df

In [152]:
# ============================================================
# Step 11: Function to save one airport parquet file
# ------------------------------------------------------------
# This function saves one parquet file per airport.
#
# Naming files by airport code makes the workflow easy to:
# - inspect,
# - rerun,
# - debug,
# - load later.
# ============================================================

def save_airport_parquet(weather_df, airport_code, output_dir):
    file_path = output_dir / f"airport_{airport_code}.parquet"
    weather_df.to_parquet(file_path, index=False)
    return file_path

In [153]:
# ============================================================
# Step 12: Helper to check whether a parquet file
# already exists for an airport
# ------------------------------------------------------------
# This is used to safely skip completed airports when the
# notebook is rerun.
# ============================================================

def get_airport_parquet_path(airport_code, output_dir):
    return output_dir / f"airport_{airport_code}.parquet"

## Run the batch ingestion

This section loops through the selected batch of airports, calls the API for each one, writes the parquet file, and records the result in a batch log.

In [154]:
# ============================================================
# Step 13: Process the current batch of airports
# ------------------------------------------------------------
# This is the main ingestion loop.
#
# For each airport:
# - skip if file already exists and overwrite is False
# - call the API
# - save the parquet file
# - record success or failure in a results list
# - sleep briefly before the next request
# ============================================================

results = []

for _, row in batch_df.iterrows():
    airport_code = str(row["airport"]).strip()
    latitude = row["latitude"]
    longitude = row["longitude"]

    parquet_path = get_airport_parquet_path(airport_code, WEATHER_OUTPUT_DIR)

    if parquet_path.exists() and not OVERWRITE_EXISTING:
        print(f"Skipping {airport_code}: parquet file already exists")
        results.append({
            "airport": airport_code,
            "status": "skipped_existing",
            "rows_written": None,
            "file_path": str(parquet_path),
            "error_message": None
        })
        continue

    try:
        print(f"Fetching weather for airport {airport_code}...")

        weather_df = fetch_airport_weather(
            airport_code=airport_code,
            latitude=latitude,
            longitude=longitude,
            start_date=START_DATE,
            end_date=END_DATE
        )

        saved_file = save_airport_parquet(
            weather_df=weather_df,
            airport_code=airport_code,
            output_dir=WEATHER_OUTPUT_DIR
        )

        print(f"Saved: {saved_file}")

        results.append({
            "airport": airport_code,
            "status": "success",
            "rows_written": len(weather_df),
            "file_path": str(saved_file),
            "error_message": None
        })

    except Exception as e:
        print(f"Failed for {airport_code}: {e}")

        results.append({
            "airport": airport_code,
            "status": "failed",
            "rows_written": None,
            "file_path": None,
            "error_message": str(e)
        })

    # Pause between API calls to reduce pressure on the service
    time.sleep(SLEEP_SECONDS)

Fetching weather for airport TWF...
Saved: c:\Users\thanu\OneDrive\Documents\MDSA Fall 2025 to Winter 2026\Data 608\Data 608 Project\Open-Meto-Weather\weather_parquet\airport_TWF.parquet
Fetching weather for airport TXK...
Saved: c:\Users\thanu\OneDrive\Documents\MDSA Fall 2025 to Winter 2026\Data 608\Data 608 Project\Open-Meto-Weather\weather_parquet\airport_TXK.parquet
Fetching weather for airport TYR...
Saved: c:\Users\thanu\OneDrive\Documents\MDSA Fall 2025 to Winter 2026\Data 608\Data 608 Project\Open-Meto-Weather\weather_parquet\airport_TYR.parquet
Fetching weather for airport TYS...
Saved: c:\Users\thanu\OneDrive\Documents\MDSA Fall 2025 to Winter 2026\Data 608\Data 608 Project\Open-Meto-Weather\weather_parquet\airport_TYS.parquet
Fetching weather for airport USA...
Saved: c:\Users\thanu\OneDrive\Documents\MDSA Fall 2025 to Winter 2026\Data 608\Data 608 Project\Open-Meto-Weather\weather_parquet\airport_USA.parquet
Fetching weather for airport VCT...
Saved: c:\Users\thanu\OneDriv

In [155]:
# ============================================================
# Step 14: Convert results list into a batch log dataframe
# ------------------------------------------------------------
# This cell summarizes what happened during the batch run.
# ============================================================

results_df = pd.DataFrame(results)

print("Batch processing complete.")
results_df.head()

Batch processing complete.


,airport,status,rows_written,file_path,error_message
0,TWF,success,26448,c:\Users\thanu\OneDrive\Documents\MDSA Fall 20...,None
1,TXK,success,26448,c:\Users\thanu\OneDrive\Documents\MDSA Fall 20...,None
2,TYR,success,26448,c:\Users\thanu\OneDrive\Documents\MDSA Fall 20...,None
3,TYS,success,26448,c:\Users\thanu\OneDrive\Documents\MDSA Fall 20...,None
4,USA,success,26448,c:\Users\thanu\OneDrive\Documents\MDSA Fall 20...,None


In [156]:
# ============================================================
# Step 15: Save the batch log to CSV
# ------------------------------------------------------------
# This cell writes a log file so you have a permanent record
# of what happened in the current batch.
# ============================================================

batch_number = (BATCH_START // BATCH_SIZE) + 1
batch_log_file = BATCH_LOG_DIR / f"batch_{batch_number:03d}_results.csv"

results_df.to_csv(batch_log_file, index=False)

print("Batch log saved to:")
print(batch_log_file)

Batch log saved to:
c:\Users\thanu\OneDrive\Documents\MDSA Fall 2025 to Winter 2026\Data 608\Data 608 Project\Open-Meto-Weather\batch_logs\batch_008_results.csv


In [157]:
# ============================================================
# Step 16: Review batch outcome counts
# ------------------------------------------------------------
# This cell provides a quick summary of:
# - successes
# - failures
# - skipped files
# ============================================================

results_df["status"].value_counts(dropna=False)

status
success    16
Name: count, dtype: int64

In [158]:
# ============================================================
# Step 17: View failed airports only
# ------------------------------------------------------------
# This cell helps identify which airports should be retried
# later if some requests failed.
# ============================================================

failed_airports_df = results_df[results_df["status"] == "failed"].copy()
failed_airports_df

,airport,status,rows_written,file_path,error_message


In [159]:
# ============================================================
# Step 18: Read one parquet file as a validation check
# ------------------------------------------------------------
# This is a simple quality check to confirm that the saved
# parquet files can be read correctly.
# ============================================================

example_files = list(WEATHER_OUTPUT_DIR.glob("airport_*.parquet"))

if example_files:
    example_df = pd.read_parquet(example_files[0])
    print("Example parquet file:", example_files[0].name)
    display(example_df.head())
else:
    print("No parquet files found yet.")

Example parquet file: airport_ABE.parquet


,datetime_utc,datetime_local,datetime_utc_str,datetime_local_str,temperature_2m,wind_speed_10m,wind_gusts_10m,precipitation,rain,snowfall,weather_code,airport,latitude,longitude,timezone,timezone_abbreviation,utc_offset_seconds,elevation
0,2022-10-29 04:00:00,2022-10-29 00:00:00,2022-10-29 04:00:00,2022-10-29 00:00:00,3.35,7.594207,10.080000,0.0,0.0,0.0,0.0,ABE,40.651667,-75.442778,b'America/New_York',b'GMT-4',-14400,117.0
1,2022-10-29 05:00:00,2022-10-29 01:00:00,2022-10-29 05:00:00,2022-10-29 01:00:00,2.65,7.636753,10.799999,0.0,0.0,0.0,0.0,ABE,40.651667,-75.442778,b'America/New_York',b'GMT-4',-14400,117.0
2,2022-10-29 06:00:00,2022-10-29 02:00:00,2022-10-29 06:00:00,2022-10-29 02:00:00,2.10,7.636753,10.799999,0.0,0.0,0.0,0.0,ABE,40.651667,-75.442778,b'America/New_York',b'GMT-4',-14400,117.0
3,2022-10-29 07:00:00,2022-10-29 03:00:00,2022-10-29 07:00:00,2022-10-29 03:00:00,1.90,7.208994,10.799999,0.0,0.0,0.0,0.0,ABE,40.651667,-75.442778,b'America/New_York',b'GMT-4',-14400,117.0
4,2022-10-29 08:00:00,2022-10-29 04:00:00,2022-10-29 08:00:00,2022-10-29 04:00:00,1.65,7.200000,10.080000,0.0,0.0,0.0,0.0,ABE,40.651667,-75.442778,b'America/New_York',b'GMT-4',-14400,117.0


## How to move to the next batch

To process the next group of airports:
1. Change `BATCH_START`
2. Keep `BATCH_SIZE = 50`
3. Run the notebook again

### Example
- Batch 1: `BATCH_START = 0`
- Batch 2: `BATCH_START = 50`
- Batch 3: `BATCH_START = 100`

Because the notebook saves one parquet file per airport and skips existing files, reruns are safe.